<a href="https://colab.research.google.com/github/JawadAhmadAnsari/SLM_fine_tuning_CUDA/blob/main/notebooks/colab_controller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Controller for SLMs Fine-Tuning

This notebook is the entry point for running the project on Google Colab. It clones the repository, installs dependencies, and drives the pipeline through `src/controller.py` (train → evaluate → drift check).

## 1. Clone Repository & Set Working Directory

Clones the project and changes into the project root. A `GIT_TOKEN` secret is used only if present (needed for a private repo).

In [ ]:
from google.colab import userdata

try:
    git_token = userdata.get('GIT_TOKEN')
except Exception:
    git_token = None

if git_token:
    !git clone https://{git_token}@github.com/JawadAhmadAnsari/SLM_fine_tuning_CUDA.git
else:
    !git clone https://github.com/JawadAhmadAnsari/SLM_fine_tuning_CUDA.git

%cd SLM_fine_tuning_CUDA

## 2. Install Dependencies

`requirements.txt` pins a tested set of versions (torch, transformers, trl, unsloth, peft, ...).

In [ ]:
!pip install -r requirements.txt

## 3. Set Environment Variables & Secrets

Sets the output directory and loads secrets from Colab's secret manager. Every secret is optional: missing ones are skipped (e.g. without `MLFLOW_TRACKING_URI`, training runs without MLflow tracking).

In [ ]:
import os
from google.colab import userdata

# Overrides training.output_dir in configs/config.yaml
output_dir = "/content/SLM_fine_tuning_CUDA/outputs"
os.environ['OUTPUT_DIR'] = output_dir
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

SECRETS = [
    'MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD',
    'HUGGING_FACE_TOKEN', 'OPENAI_API_KEY',
    'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST',
]
for name in SECRETS:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f"Loaded {name}")
    else:
        print(f"Skipped {name} (not set)")

## 4. Train

Fine-tunes the model via the controller. The adapter is saved to `outputs/final_model`.

In [ ]:
!python -m src.controller --task train

## 5. Evaluate & Save Baseline

Scores the fine-tuned model (ROUGE-L on the held-out test split) and saves it as the drift baseline.

In [ ]:
!python -m src.controller --task eval --save_baseline

## 6. Drift Check

Re-evaluates and compares against the baseline; retrains automatically if the drop exceeds `drift.retrain_threshold` in `configs/autoeval.yaml`. Run this later (e.g. after data or model changes).

In [ ]:
!python -m src.controller --task drift

## 7. Build PakWheels Dataset

Regenerates the training set from the knowledge base. Only needed after editing the knowledge base or intents; the committed .jsonl is already up to date.

In [ ]:
!python scripts/build_pakwheels_dataset.py

## 8. Run RAG Evaluation

Requires the `OPENAI_API_KEY` secret (RAGAS uses it as the judge).

In [ ]:
!python scripts/evaluate.py

## 9. Deploy Interface

Launches the Gradio app with a public share link.

In [ ]:
!python deployment/app.py